# Section 00 — Screen activity pipeline overview

This notebook is the canonical production-shaped prototype for Screcaap's processing pipeline. It is still a notebook, not a deployed service and not an implementation of the production OCRProvider.

Current flow:

Input images → OCR → normalized OCR artifacts

Planned flow:

Input images → OCR → Semantic analysis [future] → Temporal aggregation [future] → Activity/focus logs [future]

The implemented stages emphasize correctness, observability, recoverability, deterministic ordering, incremental persistence, and clear boundaries. Sequential GPU inference is intentional until profiling demonstrates a need for concurrency.

> Research findings should be validated separately before being promoted here.

Intermediate artifacts remain visible so a later failure can be localized to input loading, OCR, semantic interpretation, or temporal aggregation.

## Section 01 — Environment and dependency setup

This section reuses the compatibility set validated by the PP-OCRv5 research notebook; it does not discover a new Paddle stack.

Validated set:

- PaddlePaddle GPU 3.2.2 from the official CUDA 11.8 wheel index
- PaddleOCR 3.7.0
- PaddleX 3.7.2 with the ocr-core extra
- PyYAML 6.0.2
- opencv-contrib-python 4.10.0.84

The install cell is idempotent. If it changes package files, stop, restart the Kaggle session once, and then use Run All. On the second pass the exact versions are verified and pip is skipped. Do not try to repair unrelated Kaggle packages such as ydata-profiling or google-adk.

In [ ]:
import importlib.metadata
import os
import platform
import shutil
import subprocess
import sys


PIPELINE_NOTEBOOK_REVISION = "2026-10-02-screen-pipeline-r1"
VALIDATED_OCR_SETUP_REVISION = "2026-10-02-paddlex-3.7.2-r5"
PADDLE_VERSION = "3.2.2"
PADDLEOCR_VERSION = "3.7.0"
PADDLEX_VERSION = "3.7.2"
PYYAML_VERSION = "6.0.2"
OPENCV_CONTRIB_VERSION = "4.10.0.84"


def installed_version(distribution_name):
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


print(f"Pipeline notebook revision: {PIPELINE_NOTEBOOK_REVISION}")
print(f"Validated OCR setup revision: {VALIDATED_OCR_SETUP_REVISION}")
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Kaggle runtime: {os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'not detected')}")

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    smi_result = subprocess.run(
        [nvidia_smi], capture_output=True, text=True, timeout=30, check=False
    )
    SYSTEM_HAS_NVIDIA_GPU = smi_result.returncode == 0
    print(f"Kaggle GPU enabled: {SYSTEM_HAS_NVIDIA_GPU}")
    print("\n--- nvidia-smi ---")
    print(smi_result.stdout if smi_result.stdout else smi_result.stderr)
else:
    SYSTEM_HAS_NVIDIA_GPU = False
    print("Kaggle GPU enabled: False")

for distribution_name in (
    "paddlepaddle",
    "paddlepaddle-gpu",
    "paddleocr",
    "paddlex",
    "PyYAML",
    "numpy",
    "opencv-contrib-python",
    "protobuf",
    "torch",
    "nvidia-nccl-cu12",
):
    print(f"{distribution_name}: {installed_version(distribution_name)}")

In [ ]:
cpu_paddle = installed_version("paddlepaddle")
gpu_paddle = installed_version("paddlepaddle-gpu")
if cpu_paddle != "not installed" and gpu_paddle != "not installed":
    raise RuntimeError(
        "Both paddlepaddle and paddlepaddle-gpu are installed. Start a fresh Kaggle "
        "session before continuing; Paddle does not support both builds together."
    )

if SYSTEM_HAS_NVIDIA_GPU:
    paddle_distribution = "paddlepaddle-gpu"
    paddle_index = "https://www.paddlepaddle.org.cn/packages/stable/cu118/"
    conflicting_distribution = "paddlepaddle"
else:
    paddle_distribution = "paddlepaddle"
    paddle_index = "https://www.paddlepaddle.org.cn/packages/stable/cpu/"
    conflicting_distribution = "paddlepaddle-gpu"

if installed_version(conflicting_distribution) != "not installed":
    raise RuntimeError(
        f"{conflicting_distribution} is installed, but this runtime needs "
        f"{paddle_distribution}. Start a fresh Kaggle session."
    )

paddle_needs_install = installed_version(paddle_distribution) != PADDLE_VERSION
ocr_needs_install = any(
    [
        installed_version("paddleocr") != PADDLEOCR_VERSION,
        installed_version("paddlex") != PADDLEX_VERSION,
        installed_version("PyYAML") != PYYAML_VERSION,
        installed_version("opencv-contrib-python") != OPENCV_CONTRIB_VERSION,
    ]
)
INSTALLATION_CHANGED = paddle_needs_install or ocr_needs_install

if paddle_needs_install:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "--no-cache-dir",
            "--upgrade-strategy",
            "only-if-needed",
            f"{paddle_distribution}=={PADDLE_VERSION}",
            "--index-url",
            paddle_index,
        ]
    )
else:
    print(f"{paddle_distribution} already matches {PADDLE_VERSION}; skipping installation.")

if ocr_needs_install:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "--no-cache-dir",
            "--upgrade-strategy",
            "only-if-needed",
            f"paddleocr=={PADDLEOCR_VERSION}",
            f"paddlex[ocr-core]=={PADDLEX_VERSION}",
        ]
    )
else:
    print("PaddleOCR/PaddleX and exact OCR dependencies already match; skipping installation.")

print(f"Installed {paddle_distribution}: {installed_version(paddle_distribution)}")
print(f"Installed PaddleOCR: {installed_version('paddleocr')}")
print(f"Installed PaddleX: {installed_version('paddlex')}")
print(f"Installed PyYAML: {installed_version('PyYAML')}")
print(f"Installed OpenCV contrib: {installed_version('opencv-contrib-python')}")

if INSTALLATION_CHANGED:
    print(
        "\nSTOP: package files changed. Restart the Kaggle session once, then use Run All. "
        "The verification section will refuse to import from this mutated process."
    )
else:
    print("No package files changed; continue to environment verification.")

## Section 02 — Environment verification

The exact imports are tested first in a fresh subprocess so installed-file failures are distinguishable from stale notebook state. Original subprocess stdout and stderr are preserved.

Kaggle's base image includes PyTorch, but OCR does not need it. The validated r5 strategy temporarily disables ModelScope's optional Torch discovery only while importing PaddleX, restores the environment/import hook immediately, and verifies that libtorch_cuda.so was not loaded. This avoids the observed NCCL symbol collision without uninstalling Torch or editing site-packages.

In [ ]:
import json
import time
from contextlib import contextmanager
import importlib.util
from pathlib import Path
from packaging.version import Version


if INSTALLATION_CHANGED:
    raise RuntimeError(
        "Dependency files changed in Section 01. Restart the Kaggle session, then use Run All."
    )

expected_exact_versions = {
    paddle_distribution: PADDLE_VERSION,
    "paddleocr": PADDLEOCR_VERSION,
    "paddlex": PADDLEX_VERSION,
    "PyYAML": PYYAML_VERSION,
    "opencv-contrib-python": OPENCV_CONTRIB_VERSION,
}
version_mismatches = {
    name: (expected, installed_version(name))
    for name, expected in expected_exact_versions.items()
    if installed_version(name) != expected
}
if version_mismatches:
    raise RuntimeError(f"Pinned OCR package mismatch: {version_mismatches}")

numpy_version = installed_version("numpy")
protobuf_version = installed_version("protobuf")
if numpy_version == "not installed" or not (
    Version("1.24") <= Version(numpy_version) < Version("2.4")
):
    raise RuntimeError(f"PaddleX requires numpy>=1.24,<2.4; found {numpy_version}.")
if protobuf_version == "not installed" or Version(protobuf_version) < Version("3.20.2"):
    raise RuntimeError(f"PaddlePaddle requires protobuf>=3.20.2; found {protobuf_version}.")

smoke_code = r'''
from contextlib import contextmanager
import importlib.metadata as metadata
import importlib.util
import os
from pathlib import Path
import sys

@contextmanager
def isolate_optional_torch_from_modelscope():
    original_find_spec = importlib.util.find_spec
    original_use_torch = os.environ.get("USE_TORCH")

    def find_spec_without_torch(name, package=None):
        if name == "torch" or name.startswith("torch."):
            return None
        return original_find_spec(name, package)

    os.environ["USE_TORCH"] = "0"
    importlib.util.find_spec = find_spec_without_torch
    try:
        yield
    finally:
        importlib.util.find_spec = original_find_spec
        if original_use_torch is None:
            os.environ.pop("USE_TORCH", None)
        else:
            os.environ["USE_TORCH"] = original_use_torch

def native_library_loaded(library_name):
    process_maps = Path("/proc/self/maps")
    return process_maps.is_file() and library_name in process_maps.read_text(errors="replace")

for name in ("paddlepaddle", "paddlepaddle-gpu", "paddleocr", "paddlex", "PyYAML", "numpy", "opencv-contrib-python", "protobuf", "torch", "nvidia-nccl-cu12"):
    try:
        version = metadata.version(name)
    except metadata.PackageNotFoundError:
        version = "not installed"
    print(f"{name}: {version}", flush=True)

with isolate_optional_torch_from_modelscope():
    import paddle
    from paddleocr import PaddleOCR
    import paddleocr
    import paddlex

libtorch_cuda_loaded = native_library_loaded("libtorch_cuda.so")
print(f"libtorch_cuda.so loaded: {libtorch_cuda_loaded}", flush=True)
if libtorch_cuda_loaded:
    raise RuntimeError("ModelScope isolation failed: libtorch_cuda.so was loaded.")
print(f"CUDA compiled: {bool(paddle.is_compiled_with_cuda())}", flush=True)
print(f"GPU count: {int(paddle.device.cuda.device_count()) if paddle.is_compiled_with_cuda() else 0}", flush=True)
print("IMPORT_SMOKE_TEST_OK", flush=True)
'''

smoke_environment = os.environ.copy()
smoke_environment.pop("PADDLE_PDX_EAGER_INIT", None)
smoke_result = subprocess.run(
    [sys.executable, "-c", smoke_code],
    env=smoke_environment,
    capture_output=True,
    text=True,
    check=False,
)
print("--- fresh-process stdout ---")
print(smoke_result.stdout)
if smoke_result.stderr:
    print("--- fresh-process stderr ---", file=sys.stderr)
    print(smoke_result.stderr, file=sys.stderr)
if smoke_result.returncode != 0:
    raise subprocess.CalledProcessError(
        smoke_result.returncode,
        smoke_result.args,
        output=smoke_result.stdout,
        stderr=smoke_result.stderr,
    )

stale_import_modules = sorted(
    name
    for name in sys.modules
    if any(
        name == prefix or name.startswith(f"{prefix}.")
        for prefix in ("paddle", "paddleocr", "paddlex", "modelscope")
    )
)
if stale_import_modules:
    raise RuntimeError(
        "Fresh-process imports passed, but this kernel already contains OCR modules: "
        f"{stale_import_modules[:8]}. Restart Session and use Run All."
    )

@contextmanager
def isolate_optional_torch_from_modelscope():
    original_find_spec = importlib.util.find_spec
    original_use_torch = os.environ.get("USE_TORCH")

    def find_spec_without_torch(name, package=None):
        if name == "torch" or name.startswith("torch."):
            return None
        return original_find_spec(name, package)

    os.environ["USE_TORCH"] = "0"
    importlib.util.find_spec = find_spec_without_torch
    try:
        yield
    finally:
        importlib.util.find_spec = original_find_spec
        if original_use_torch is None:
            os.environ.pop("USE_TORCH", None)
        else:
            os.environ["USE_TORCH"] = original_use_torch


def native_library_loaded(library_name):
    process_maps = Path("/proc/self/maps")
    return process_maps.is_file() and library_name in process_maps.read_text(errors="replace")


if native_library_loaded("libtorch_cuda.so"):
    raise RuntimeError("libtorch_cuda.so is already loaded. Restart Session and use Run All.")

os.environ.pop("PADDLE_PDX_EAGER_INIT", None)
with isolate_optional_torch_from_modelscope():
    import paddle
    from paddleocr import PaddleOCR
    import paddleocr
    import paddlex

if native_library_loaded("libtorch_cuda.so"):
    raise RuntimeError("ModelScope isolation failed: libtorch_cuda.so was loaded.")

import numpy as np
from IPython.display import display
from PIL import Image, UnidentifiedImageError

PADDLE_HAS_CUDA = bool(paddle.is_compiled_with_cuda())
PADDLE_GPU_COUNT = int(paddle.device.cuda.device_count()) if PADDLE_HAS_CUDA else 0
PADDLE_DEVICE = paddle.device.get_device()

print("Environment verification: OK")
print(f"  Python: {sys.version.split()[0]}")
print(f"  Platform: {platform.platform()}")
print(f"  PaddlePaddle: {paddle.__version__}")
print(f"  PaddleOCR: {paddleocr.__version__}")
print(f"  PaddleX: {paddlex.__version__}")
print(f"  CUDA compiled: {PADDLE_HAS_CUDA}")
print(f"  GPU count: {PADDLE_GPU_COUNT}")
print(f"  Current Paddle device: {PADDLE_DEVICE}")

## Section 03 — Central pipeline configuration

Change runtime behavior only in this cell. MAX_IMAGES limits deterministic discovery for a smoke run; use None for all images. RESUME_RUN_ID must be set explicitly to reuse an existing run directory. RESUME=True with no run ID starts a new resumable run rather than guessing which earlier run to use.

In [ ]:
DEVICE = "gpu"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/screcaap")

MAX_IMAGES = None
CONTINUE_ON_ERROR = True
RESUME = True
RESUME_RUN_ID = None  # Example: "20261002T230000Z_ppocrv5_mobile"

DETECTION_MODEL = "PP-OCRv5_mobile_det"
RECOGNITION_MODEL = "latin_PP-OCRv5_mobile_rec"
OCR_VERSION = "PP-OCRv5"

if MAX_IMAGES is not None and (not isinstance(MAX_IMAGES, int) or MAX_IMAGES < 1):
    raise ValueError("MAX_IMAGES must be None or a positive integer.")
if DEVICE == "gpu":
    EFFECTIVE_DEVICE = "gpu:0"
elif DEVICE == "cpu":
    EFFECTIVE_DEVICE = "cpu"
elif DEVICE.startswith("gpu:"):
    EFFECTIVE_DEVICE = DEVICE
else:
    raise ValueError("DEVICE must be 'cpu', 'gpu', or an explicit value such as 'gpu:0'.")

if EFFECTIVE_DEVICE.startswith("gpu"):
    if not SYSTEM_HAS_NVIDIA_GPU:
        raise RuntimeError("GPU inference requested, but Kaggle has no enabled NVIDIA accelerator.")
    if not PADDLE_HAS_CUDA or PADDLE_GPU_COUNT < 1:
        raise RuntimeError("GPU inference requested, but Paddle cannot see a CUDA device.")

print(f"Input root: {INPUT_ROOT}")
print(f"Work root: {WORK_ROOT}")
print(f"Device: {EFFECTIVE_DEVICE}")
print(f"Image limit: {MAX_IMAGES if MAX_IMAGES is not None else 'all'}")

## Section 04 — Deterministic input discovery

Images are read directly from the configured Kaggle input path; datasets do not belong in the repository. Discovery supports PNG, JPG, JPEG, and WebP. Paths are sorted by their relative POSIX path before sequence numbers are assigned, so processing never depends on arbitrary filesystem ordering.

The generated manifest is intentionally small: sample_id, sequence_index, and image_path. A later loader can replace this with an external manifest without changing downstream OCR functions.

In [ ]:
SUPPORTED_IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp"}

if not INPUT_ROOT.exists() or not INPUT_ROOT.is_dir():
    raise FileNotFoundError(
        f"INPUT_ROOT does not exist or is not a directory: {INPUT_ROOT}. "
        "Attach a Kaggle dataset or update Section 03."
    )

all_image_paths = sorted(
    (
        path
        for path in INPUT_ROOT.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_SUFFIXES
    ),
    key=lambda path: path.relative_to(INPUT_ROOT).as_posix(),
)
selected_image_paths = all_image_paths[:MAX_IMAGES] if MAX_IMAGES is not None else all_image_paths

input_manifest = [
    {
        "sample_id": f"screen_{sequence_index:06d}",
        "sequence_index": sequence_index,
        "image_path": str(path),
    }
    for sequence_index, path in enumerate(selected_image_paths, start=1)
]

print(f"Discovered: {len(all_image_paths)} image(s)")
print(f"Selected: {len(input_manifest)} image(s)")
for sample in input_manifest[:10]:
    print(
        f"  {sample['sequence_index']:06d}  {sample['sample_id']}  "
        f"{Path(sample['image_path']).relative_to(INPUT_ROOT)}"
    )
if len(input_manifest) > 10:
    print(f"  ... {len(input_manifest) - 10} more")
if not input_manifest:
    raise FileNotFoundError(f"No supported images were found under {INPUT_ROOT}.")

## Section 05 — Input validation

Validation checks existence, extension, unique sample IDs, unique sequence indices, and whether Pillow can decode each image. Invalid images are reported and retained in the run as error records; they do not prevent valid images from continuing.

In [ ]:
def validate_input_manifest(manifest):
    seen_sample_ids = set()
    seen_sequence_indices = set()
    valid = []
    invalid = []

    for sample in manifest:
        problems = []
        sample_id = sample["sample_id"]
        sequence_index = sample["sequence_index"]
        image_path = Path(sample["image_path"])

        if sample_id in seen_sample_ids:
            problems.append("duplicate sample_id")
        if sequence_index in seen_sequence_indices:
            problems.append("duplicate sequence_index")
        seen_sample_ids.add(sample_id)
        seen_sequence_indices.add(sequence_index)

        if not image_path.exists() or not image_path.is_file():
            problems.append("image does not exist")
        elif image_path.suffix.lower() not in SUPPORTED_IMAGE_SUFFIXES:
            problems.append(f"unsupported extension: {image_path.suffix}")
        else:
            try:
                with Image.open(image_path) as image:
                    image.verify()
                with Image.open(image_path) as image:
                    width, height = image.size
                    mode = image.mode
                if width < 1 or height < 1:
                    problems.append("image has invalid dimensions")
            except (OSError, UnidentifiedImageError, ValueError) as exc:
                problems.append(f"{type(exc).__name__}: {exc}")

        if problems:
            invalid.append({**sample, "validation_errors": problems})
        else:
            valid.append(
                {
                    **sample,
                    "image_width": int(width),
                    "image_height": int(height),
                    "image_mode": str(mode),
                }
            )

    return valid, invalid


valid_samples, invalid_samples = validate_input_manifest(input_manifest)
invalid_by_sample_id = {sample["sample_id"]: sample for sample in invalid_samples}

print(f"Total selected: {len(input_manifest)}")
print(f"Valid images: {len(valid_samples)}")
print(f"Invalid images: {len(invalid_samples)}")
for sample in invalid_samples:
    print(f"  {sample['sample_id']} — {'; '.join(sample['validation_errors'])}")
if not valid_samples:
    raise RuntimeError("No valid image remains after input validation.")

## Section 06 — Initialize PP-OCRv5 once

The validated screenshot configuration uses PP-OCRv5_mobile_det and latin_PP-OCRv5_mobile_rec. Document orientation classification, document unwarping, and text-line orientation classification stay disabled because desktop screenshots are already flat and upright.

Model initialization is timed separately and performed exactly once. The first initialization may download model files, so initialization_ms is not treated as per-image inference latency.

In [ ]:
initialization_started = time.perf_counter()
OCR_ENGINE = PaddleOCR(
    ocr_version=OCR_VERSION,
    text_detection_model_name=DETECTION_MODEL,
    text_recognition_model_name=RECOGNITION_MODEL,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    device=EFFECTIVE_DEVICE,
    engine="paddle_static",
)
INITIALIZATION_MS = (time.perf_counter() - initialization_started) * 1000.0

print(f"Detector: {DETECTION_MODEL}")
print(f"Recognizer: {RECOGNITION_MODEL}")
print(f"Device: {EFFECTIVE_DEVICE}")
print(f"Initialization: {INITIALIZATION_MS:.1f} ms")

## Section 07 — Single-image smoke test

The first valid image passes through the same low-level OCR normalization used by the batch boundary. The image, normalized blocks, approximate combined text, and latency are shown before any batch run begins. OCR text is noisy evidence: high confidence is not a guarantee of correctness.

In [ ]:
def to_builtin(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    return value


def public_result_payload(result):
    if not hasattr(result, "json"):
        raise TypeError("PaddleOCR result does not expose the documented public .json property.")
    data = result.json
    if callable(data):
        data = data()
    if isinstance(data, str):
        data = json.loads(data)
    data = to_builtin(data)
    if not isinstance(data, dict):
        raise TypeError(f"Expected result.json to produce a dict, got {type(data).__name__}.")
    payload = data.get("res", data)
    if not isinstance(payload, dict):
        raise TypeError("PaddleOCR public JSON does not contain a dictionary payload.")
    return payload


def polygon_bounds(polygon):
    xs = [point[0] for point in polygon]
    ys = [point[1] for point in polygon]
    return min(xs), min(ys), max(xs), max(ys)


def normalize_ocr_results(raw_results, image_width, image_height):
    blocks = []
    for payload_index, result in enumerate(raw_results):
        payload = public_result_payload(result)
        texts = list(payload.get("rec_texts") or [])
        scores = list(payload.get("rec_scores") or [])
        polygons = payload.get("rec_polys")
        if polygons is None:
            polygons = payload.get("dt_polys")
        polygons = list(polygons or [])

        if not (len(texts) == len(scores) == len(polygons)):
            raise RuntimeError(
                "PaddleOCR returned mismatched text, score, and polygon counts for "
                f"payload {payload_index}: {len(texts)}, {len(scores)}, {len(polygons)}."
            )

        for text_value, score_value, polygon_value in zip(texts, scores, polygons):
            polygon = [[float(point[0]), float(point[1])] for point in polygon_value]
            if len(polygon) != 4:
                raise RuntimeError(f"Expected a four-point polygon, got {len(polygon)} points.")
            blocks.append(
                {
                    "text": str(text_value),
                    "confidence": float(score_value),
                    "polygon": polygon,
                }
            )

    reading_order = sorted(
        range(len(blocks)),
        key=lambda index: (
            polygon_bounds(blocks[index]["polygon"])[1],
            polygon_bounds(blocks[index]["polygon"])[0],
        ),
    )
    combined_text = "\n".join(
        blocks[index]["text"]
        for index in reading_order
        if blocks[index]["text"].strip()
    )
    return {
        "engine": "paddleocr",
        "detector": DETECTION_MODEL,
        "recognizer": RECOGNITION_MODEL,
        "image_width": int(image_width),
        "image_height": int(image_height),
        "blocks": blocks,
        "combined_text": combined_text,
    }


def synchronize_device():
    if EFFECTIVE_DEVICE.startswith("gpu"):
        paddle.device.synchronize()


def predict_and_normalize(image_path, image_width, image_height):
    synchronize_device()
    started = time.perf_counter()
    raw_results = list(OCR_ENGINE.predict(str(image_path)))
    synchronize_device()
    elapsed_ms = (time.perf_counter() - started) * 1000.0
    if not raw_results:
        raise RuntimeError("PaddleOCR returned no result object.")
    normalized = normalize_ocr_results(raw_results, image_width, image_height)
    return normalized, elapsed_ms


smoke_sample = valid_samples[0]
with Image.open(smoke_sample["image_path"]) as image:
    smoke_image = image.copy()
display(smoke_image)

smoke_ocr, smoke_latency_ms = predict_and_normalize(
    Path(smoke_sample["image_path"]),
    smoke_sample["image_width"],
    smoke_sample["image_height"],
)
print(f"Sample: {smoke_sample['sample_id']}")
print(f"Latency: {smoke_latency_ms:.1f} ms")
print(f"OCR blocks: {len(smoke_ocr['blocks'])}")
print(json.dumps(smoke_ocr["blocks"][:10], ensure_ascii=False, indent=2))
print("\nCombined text:\n")
print(smoke_ocr["combined_text"] or "[No recognized text]")

## Section 08 — Normalized OCR contract

The project-neutral OCR artifact preserves engine/model identity, image dimensions, every recognized text block, confidence, polygon geometry, and approximate combined text. It deliberately does not filter suspicious characters, repair Vietnamese, or interpret user activity.

OCR confidence describes the model's own score; it is not a calibrated probability that the text is correct. Downstream stages must retain access to original OCR evidence.

In [ ]:
def validate_ocr_contract(ocr_result):
    required_keys = {
        "engine",
        "detector",
        "recognizer",
        "image_width",
        "image_height",
        "blocks",
        "combined_text",
    }
    missing = required_keys.difference(ocr_result)
    if missing:
        raise ValueError(f"OCR contract is missing fields: {sorted(missing)}")
    if not isinstance(ocr_result["blocks"], list):
        raise TypeError("ocr.blocks must be a list.")
    for index, block in enumerate(ocr_result["blocks"]):
        if not isinstance(block.get("text"), str):
            raise TypeError(f"ocr.blocks[{index}].text must be a string.")
        if not isinstance(block.get("confidence"), (int, float)):
            raise TypeError(f"ocr.blocks[{index}].confidence must be numeric.")
        polygon = block.get("polygon")
        if not isinstance(polygon, list) or len(polygon) != 4:
            raise ValueError(f"ocr.blocks[{index}].polygon must have four points.")
    return True


validate_ocr_contract(smoke_ocr)
print("Normalized OCR contract: OK")

## Section 09 — Sample OCR processing boundary

process_ocr(sample) owns exactly one stage:

load image → OCR → normalize → timings → structured result

It contains no semantic classification, temporal inference, persistence policy, or provider-specific behavior outside the OCR adapter call. This boundary allows a future process_semantic(ocr_result) stage to be attached independently.

In [ ]:
def process_ocr(sample):
    image_path = Path(sample["image_path"])
    with Image.open(image_path) as image:
        image.load()
        width, height = image.size

    ocr_result, ocr_ms = predict_and_normalize(image_path, width, height)
    validate_ocr_contract(ocr_result)
    return {
        "sample_id": sample["sample_id"],
        "sequence_index": int(sample["sequence_index"]),
        "status": "ok",
        "image": {
            "path": str(image_path),
            "width": int(width),
            "height": int(height),
        },
        "ocr": ocr_result,
        "timing": {"ocr_ms": float(ocr_ms)},
    }


print("process_ocr boundary defined; OCR_ENGINE remains the single initialized model instance.")

## Section 10 — Sequential batch processing

The initial batch runner is intentionally sequential. Each sample is persisted immediately, and one failure does not stop later samples when CONTINUE_ON_ERROR is true. Concurrency should be introduced only after profiling demonstrates a bottleneck and recovery semantics remain clear.

In [ ]:
def append_jsonl_and_flush(file_handle, record):
    file_handle.write(json.dumps(to_builtin(record), ensure_ascii=False) + "\n")
    file_handle.flush()
    os.fsync(file_handle.fileno())


def process_batch(samples, results_jsonl_path, successful_sample_ids, invalid_samples_by_id):
    total = len(samples)
    with results_jsonl_path.open("a", encoding="utf-8") as results_file:
        for position, sample in enumerate(samples, start=1):
            sample_id = sample["sample_id"]
            if sample_id in successful_sample_ids:
                print(f"[{position}/{total}] {sample_id} — SKIP — already successful")
                continue

            invalid_sample = invalid_samples_by_id.get(sample_id)
            if invalid_sample is not None:
                record = {
                    "sample_id": sample_id,
                    "sequence_index": int(sample["sequence_index"]),
                    "status": "error",
                    "image": {"path": sample["image_path"]},
                    "error": {
                        "type": "InputValidationError",
                        "message": "; ".join(invalid_sample["validation_errors"]),
                    },
                }
                append_jsonl_and_flush(results_file, record)
                print(f"[{position}/{total}] {sample_id} — ERROR — input validation")
                if not CONTINUE_ON_ERROR:
                    raise RuntimeError(record["error"]["message"])
                continue

            try:
                record = process_ocr(sample)
                append_jsonl_and_flush(results_file, record)
                print(
                    f"[{position}/{total}] {sample_id} — OK — "
                    f"{record['timing']['ocr_ms']:.1f} ms — "
                    f"{len(record['ocr']['blocks'])} OCR blocks"
                )
            except Exception as exc:
                record = {
                    "sample_id": sample_id,
                    "sequence_index": int(sample["sequence_index"]),
                    "status": "error",
                    "image": {"path": sample["image_path"]},
                    "error": {"type": type(exc).__name__, "message": str(exc)},
                }
                append_jsonl_and_flush(results_file, record)
                print(f"[{position}/{total}] {sample_id} — ERROR — {type(exc).__name__}: {exc}")
                if not CONTINUE_ON_ERROR:
                    raise


print("Sequential batch runner defined; execution begins only after run persistence is prepared.")

## Section 11 — Run structure, incremental persistence, and resume

Every run lives under /kaggle/working/screcaap/runs/<run_id>. New run IDs are UTC timestamps plus ppocrv5_mobile.

Resume selection is explicit:

- Set RESUME=True and RESUME_RUN_ID to an existing directory name to continue that run.
- Set RESUME=True and RESUME_RUN_ID=None to create a new resumable run.
- Failed rows are retried; only successful sample IDs are skipped.
- The manifest fingerprint and critical configuration must match before an existing run is resumed.

results.jsonl is the append-only execution history and is flushed after every sample. results.json contains the latest row per sample, ordered by sequence_index. If a failed sample later succeeds, both attempts remain in JSONL while the latest success becomes canonical. errors.jsonl contains unresolved latest failures.

In [ ]:
import hashlib
import math
from datetime import datetime, timezone
from statistics import mean, median


def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()


def load_jsonl(path):
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as input_file:
        for line_number, line in enumerate(input_file, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSONL at {path}:{line_number}: {exc}") from exc
            if not isinstance(row, dict):
                raise TypeError(f"Expected an object at {path}:{line_number}.")
            rows.append(row)
    return rows


def atomic_write_json(path, value):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as output_file:
        json.dump(to_builtin(value), output_file, ensure_ascii=False, indent=2)
        output_file.flush()
        os.fsync(output_file.fileno())
    temporary_path.replace(path)


def atomic_write_jsonl(path, rows):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as output_file:
        for row in rows:
            output_file.write(json.dumps(to_builtin(row), ensure_ascii=False) + "\n")
        output_file.flush()
        os.fsync(output_file.fileno())
    temporary_path.replace(path)


def percentile(values, fraction):
    if not values:
        return None
    ordered = sorted(float(value) for value in values)
    if len(ordered) == 1:
        return ordered[0]
    position = (len(ordered) - 1) * fraction
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    return ordered[lower] + (ordered[upper] - ordered[lower]) * (position - lower)


manifest_identity = [
    {
        "sample_id": sample["sample_id"],
        "sequence_index": sample["sequence_index"],
        "relative_path": Path(sample["image_path"]).relative_to(INPUT_ROOT).as_posix(),
    }
    for sample in input_manifest
]
manifest_fingerprint = hashlib.sha256(
    json.dumps(manifest_identity, ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest()

run_configuration = {
    "input_root": str(INPUT_ROOT),
    "max_images": MAX_IMAGES,
    "device": EFFECTIVE_DEVICE,
    "detector": DETECTION_MODEL,
    "recognizer": RECOGNITION_MODEL,
    "continue_on_error": CONTINUE_ON_ERROR,
    "manifest_fingerprint": manifest_fingerprint,
}
runs_root = WORK_ROOT / "runs"
runs_root.mkdir(parents=True, exist_ok=True)

if RESUME_RUN_ID is not None and not RESUME:
    raise ValueError("RESUME_RUN_ID requires RESUME=True.")

if RESUME_RUN_ID is not None:
    RUN_ID = RESUME_RUN_ID
    run_directory = runs_root / RUN_ID
    run_path = run_directory / "run.json"
    if not run_path.exists():
        raise FileNotFoundError(f"Cannot resume; run.json does not exist: {run_path}")
    with run_path.open("r", encoding="utf-8") as input_file:
        run_metadata = json.load(input_file)
    previous_configuration = run_metadata.get("configuration")
    if previous_configuration != run_configuration:
        raise RuntimeError(
            "Resume configuration or manifest does not match run.json. "
            "Start a new run instead of mixing incompatible results."
        )
    print(f"Resuming explicit run: {RUN_ID}")
else:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_ppocrv5_mobile")
    run_directory = runs_root / RUN_ID
    run_directory.mkdir(parents=True, exist_ok=False)
    run_metadata = {
        "run_id": RUN_ID,
        "started_at": utc_now_iso(),
        "pipeline_notebook_revision": PIPELINE_NOTEBOOK_REVISION,
        "validated_ocr_setup_revision": VALIDATED_OCR_SETUP_REVISION,
        "python_version": sys.version.split()[0],
        "platform": platform.platform(),
        "paddle_distribution": paddle_distribution,
        "paddle_version": installed_version(paddle_distribution),
        "paddleocr_version": installed_version("paddleocr"),
        "paddlex_version": installed_version("paddlex"),
        "device": EFFECTIVE_DEVICE,
        "detector": DETECTION_MODEL,
        "recognizer": RECOGNITION_MODEL,
        "input_count": len(input_manifest),
        "configuration": run_configuration,
    }
    atomic_write_json(run_directory / "run.json", run_metadata)
    print(f"Created new run: {RUN_ID}")

results_jsonl_path = run_directory / "results.jsonl"
results_jsonl_path.touch(exist_ok=True)
existing_rows = load_jsonl(results_jsonl_path)
successful_sample_ids = {
    row["sample_id"]
    for row in existing_rows
    if row.get("status") == "ok" and isinstance(row.get("sample_id"), str)
}

process_batch(
    input_manifest,
    results_jsonl_path,
    successful_sample_ids,
    invalid_by_sample_id,
)

all_history_rows = load_jsonl(results_jsonl_path)
latest_by_sample_id = {}
for row in all_history_rows:
    latest_by_sample_id[row["sample_id"]] = row
selected_sample_ids = {sample["sample_id"] for sample in input_manifest}
ordered_results = sorted(
    (row for sample_id, row in latest_by_sample_id.items() if sample_id in selected_sample_ids),
    key=lambda row: int(row["sequence_index"]),
)

error_rows = [
    {
        "sample_id": row["sample_id"],
        "sequence_index": row["sequence_index"],
        "image_path": (row.get("image") or {}).get("path"),
        "error_type": (row.get("error") or {}).get("type"),
        "error_message": (row.get("error") or {}).get("message"),
    }
    for row in ordered_results
    if row.get("status") == "error"
]
success_rows = [row for row in ordered_results if row.get("status") == "ok"]
latencies_ms = [row["timing"]["ocr_ms"] for row in success_rows]

summary = {
    "run_id": RUN_ID,
    "total": len(input_manifest),
    "successful": len(success_rows),
    "failed": len(error_rows),
    "mean_ocr_ms": mean(latencies_ms) if latencies_ms else None,
    "p50_ocr_ms": median(latencies_ms) if latencies_ms else None,
    "p95_ocr_ms": percentile(latencies_ms, 0.95),
    "min_ocr_ms": min(latencies_ms) if latencies_ms else None,
    "max_ocr_ms": max(latencies_ms) if latencies_ms else None,
    "total_ocr_blocks": sum(len(row["ocr"]["blocks"]) for row in success_rows),
    "total_recognized_characters": sum(
        len(row["ocr"]["combined_text"]) for row in success_rows
    ),
    "completed_at": utc_now_iso(),
}

atomic_write_json(run_directory / "results.json", ordered_results)
atomic_write_jsonl(run_directory / "errors.jsonl", error_rows)
atomic_write_json(run_directory / "summary.json", summary)
run_metadata["last_completed_at"] = summary["completed_at"]
atomic_write_json(run_directory / "run.json", run_metadata)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print(f"Run directory: {run_directory}")
print(f"Incremental source: {results_jsonl_path}")

## Section 12 — SEMANTIC STAGE — NOT YET PROMOTED FROM RESEARCH

This section defines only a future boundary. It does not call OpenRouter, load an API key, select an LLM, or classify activity.

A future semantic result may contain a domain/activity label, general context, application, confidence, supporting evidence, and uncertainties. The exact production ActivityAnalyzer contract remains a documented TODO and must not be decided implicitly by this notebook.

In [ ]:
SEMANTIC_RESULT_EXAMPLE = {
    "domain": "work",
    "general_context": "Editing code for an OCR project",
    "application": "Visual Studio Code",
    "confidence": 0.0,
    "evidence": [],
    "uncertainties": [],
}


def analyze_semantics(ocr_result):
    raise NotImplementedError(
        "Semantic analysis has not been promoted from research. "
        "Use research/semantic/notebooks/01_openrouter_semantic_baseline.ipynb."
    )


print("Semantic interface placeholder defined; no external API was called.")

## Section 13 — TEMPORAL AGGREGATION — FUTURE

Future flow:

ordered screenshot observations → semantic observations → activity sessions → focus/distraction patterns → user-facing logs

A screenshot is a point observation, not evidence that an activity continued until the next timestamp. Temporal rules, gap thresholds, confidence handling, and focus scoring remain unresolved product/domain decisions. This notebook does not implement them.

In [ ]:
def aggregate_temporally(ordered_semantic_observations):
    raise NotImplementedError(
        "Temporal aggregation is a future stage and must be validated separately."
    )


print("Temporal interface placeholder defined; no focus or distraction scoring was performed.")